In [1]:
import os
import sys
import torch
import tiktoken

project_root = os.path.dirname(os.path.abspath("")) # since notebook is in evaluation/
sys.path.insert(0, project_root)

import model
from model.model import GPT, GPTConfig
model.GPTConfig = GPTConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
print(f"Using device: {device}")

Using device: mps


In [3]:
# Load configuration matching the notebook setup
config = GPTConfig(
    block_size=256,
    vocab_size=50257,
    n_layer=4,
    n_head=4,
    n_embd=32,
    dropout=0.1
)

# Initialize model
model = GPT(config)
model.to(device)
model.eval()

model_path = os.path.join(project_root, "training", "nanogpt_checkpoint_ffn_reduced.pt")
if not os.path.exists(model_path):
    print(f"Error: {model_path} not found.")
    print("Please run the notebook 'training/training_pipeline.ipynb' to train and save the model.")
else:
    # Load weights
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    if "model" in checkpoint:
        model.load_state_dict(checkpoint["model"])
    else:
        model.load_state_dict(checkpoint)
    print(f"Loaded model from {model_path}")

number of parameters: 1.66M
Loaded model from /Users/idant/Developer/Projects/NanoGPT/training/nanogpt_checkpoint_ffn_reduced.pt


In [4]:
# Setup tokenizer
enc = tiktoken.get_encoding("gpt2")

prompt = "[Genre: mainstream] [Mood: aggressive] [Rhyme: internal_rhyme] [Cadence: bouncy]\nDo you like violence?"
print(f"\nPrompt: '{prompt}'\n")

idx = torch.tensor(enc.encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
max_new_tokens = 500


Prompt: '[Genre: mainstream] [Mood: aggressive] [Rhyme: internal_rhyme] [Cadence: bouncy]
Do you like violence?'



In [5]:
print("Temperature Sampling")
# Setting manual seed for determinism in generation examples
torch.manual_seed(42)
out_idx = model.generate(idx, max_new_tokens, temperature=0.7, repetition_penalty=1.2)
print(enc.decode(out_idx[0].tolist()))

Temperature Sampling
[Genre: mainstream] [Mood: aggressive] [Rhyme: internal_rhyme] [Cadence: bouncy]
Do you like violence?
You can tell me to let it came on my space"
I know, yeah!<|endoftext|>[Genre: midtempo]
Put them
How I meant to the air-oh, it's a young man feel, I'll be what they lie with us or, hey!
But listen off
And it out of me
She said that you was in my dog went back (It's have a one
I'm just need an times
Where you are some year
(Yeah, we don't no love up for him and if you're high but hard, oh)
I tell her hot
Them
These time to gettin' through the block
So nobody kind of, they wishin'all niggas, get crying in
Oh now, y'all ain't still stress
Nigga for her fly (Ouh), do better than I gotta tell you know I say how this good bitch, baby
Never got not go
My lap
Bitch)
Just stop what it you
I'ma take the best
I don't too differentnessless
Bussy and the same eye
My girl to me down, I hit your world, ooh

Ayy, so many?
Comed, baby, I don't be
The highest like I ain't be
If you